<a href="https://colab.research.google.com/github/Sparkydev007/Causal-GPT-10.7M/blob/main/Tranformers_Research_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Causal-GPT-10.7M: A Deeply Optimized Causal Transformer with Dynamic KV-Caching Inference**

An elegant, production-ready, character-level autoregressive Language Model containing 10.7 million parameters, trained completely from scratch on the Tiny Shakespeare corpus. This repository showcases advanced PyTorch systems engineering, featuring Causal Key-Value (KV) Caching for linear-time inference ($O(T)$), automated Mixed-Precision (torch.amp) acceleration, and a bulletproof, crash-resistant interactive runtime shell with aggressive out-of-vocabulary input sanitization.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import urllib.request
import time
import math

# =========================================================================
# 1. HYPERPARAMETERS & SYSTEM ENVIRONMENT CONFIGURATION
# =========================================================================
batch_size = 64
block_size = 256
max_iters = 3000         # Balanced for fast convergence (~1.49 loss) and speed
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
torch.manual_seed(1337)

print(f"🚀 SYSTEM INITIALIZATION: Running execution node on [{device.upper()}]")

# =========================================================================
# 2. DATASET INGESTION & CHARACTER-LEVEL TOKENIZATION
# =========================================================================
print("📥 Fetching Tiny Shakespeare dataset from source...")
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data_src = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_src) - block_size, (batch_size,))
    x = torch.stack([data_src[i:i+block_size] for i in ix])
    y = torch.stack([data_src[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with torch.amp.autocast(device_type=('cuda' if device == 'cuda' else 'cpu'), enabled=(device == 'cuda')):
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# =========================================================================
# 3. TRANSFORMER ARCHITECTURE MODULES WITH INFERENCE CACHING
# =========================================================================
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False):
        B, T, C = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        if use_cache and past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat((past_k, k), dim=1)
            v = torch.cat((past_v, v), dim=1)

        next_kv = (k, v) if use_cache else None
        total_T = k.shape[1]
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)

        if not (use_cache and past_kv is not None):
            wei = wei.masked_fill(self.tril[:T, :total_T] == 0, float('-inf'))

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out, next_kv

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kvs=None, use_cache=False):
        out_heads = []
        next_kvs = []
        for i, h in enumerate(self.heads):
            layer_past = past_kvs[i] if past_kvs is not None else None
            out, kv = h(x, past_kv=layer_past, use_cache=use_cache)
            out_heads.append(out)
            if use_cache:
                next_kvs.append(kv)
        out = torch.cat(out_heads, dim=-1)
        out = self.dropout(self.proj(out))
        return out, (next_kvs if use_cache else None)

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x, past_kvs=None, use_cache=False):
        sa_out, next_kv = self.sa(self.ln1(x), past_kvs=past_kvs, use_cache=use_cache)
        x = x + sa_out
        x = x + self.ffwd(self.ln2(x))
        return x, next_kv

class ProductionGPTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None, past_key_values=None, use_cache=False):
        B, T = idx.shape
        if use_cache and past_key_values is not None:
            past_length = past_key_values[0][0][0].shape[1]
            position_ids = torch.arange(past_length, past_length + T, dtype=torch.long, device=idx.device)
        else:
            position_ids = torch.arange(T, device=idx.device)

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(position_ids)
        x = tok_emb + pos_emb

        next_pkv = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            layer_past = past_key_values[i] if past_key_values is not None else None
            x, kv = block(x, past_kvs=layer_past, use_cache=use_cache)
            if use_cache:
                next_pkv.append(kv)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            return logits, next_pkv
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            return logits, loss

    @torch.inference_mode()
    def generate_cached(self, idx, max_new_tokens, temperature=1.0):
        past_key_values = None
        for i in range(max_new_tokens):
            if i == 0:
                idx_cond = idx[:, -block_size:]
                logits, past_key_values = self(idx_cond, use_cache=True)
            else:
                idx_cond = idx[:, [-1]]
                logits, past_key_values = self(idx_cond, past_key_values=past_key_values, use_cache=True)

            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=-1)
        return idx

# =========================================================================
# 4. HIGH-THROUGHPUT COMPILATION & TRAINING ENGINE
# =========================================================================
model = ProductionGPTModel()
m = model.to(device)

scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("\n🏋️ STARTING ACCELERATED TRAINING LOOP...")
start_time = time.time()

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"   Step {iter:4d}: Train Loss {losses['train']:.4f} | Val Loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    with torch.amp.autocast(device_type=('cuda' if device == 'cuda' else 'cpu'), enabled=(device == 'cuda')):
        logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

print(f"✓ Training complete in {(time.time() - start_time)/60:.2f} minutes.")

# Save pristine weight states to disk file
MODEL_FILE = "gpt_shakespeare_10.7M.pth"
torch.save(model.state_dict(), MODEL_FILE)
print(f"💾 Weights saved successfully to local directory as: ./{MODEL_FILE}")

# =========================================================================
# 5. RUN INDEPENDENT QUANTIFIABLE METRICS SWEEP
# =========================================================================
print("\n🧮 RUNNING STANDALONE PERFORMANCE SWEEP...")
model.eval()
total_val_loss = 0.0
with torch.inference_mode():
    for _ in range(100):
        X, Y = get_batch('val')
        with torch.amp.autocast(device_type=('cuda' if device == 'cuda' else 'cpu'), enabled=(device == 'cuda')):
            _, loss = model(X, Y)
        total_val_loss += loss.item()

mean_val_loss = total_val_loss / 100
perplexity = math.exp(mean_val_loss)

print("\n" + "="*50)
print("             PRODUCTION ARCHIVE METRICS REPORT")
print("="*50)
print(f"Mean Validation Loss:       {mean_val_loss:.4f}")
print(f"Model Perplexity (PPL):     {perplexity:.2f}")
print("="*50)

# =========================================================================
# 6. CRASH-PROOF INTERACTIVE SHELL INTERFACE
# =========================================================================
print("\n" + "="*50)
print("             🎯 CRASH-PROOF CHAT INTERFACE ACTIVATED")
print("============================================================\n")
print("Instructions: Type your prompt block below. Out-of-vocabulary text")
print("will be automatically sanitized. Type 'exit' to terminate.\n")

while True:
    print("-" * 60)
    raw_prompt = input("✍️ Enter Prompt: ").replace('\\n', '\n')

    if raw_prompt.strip().lower() in ['exit', 'quit']:
        print("\nExiting interactive wrapper cleanly. Work complete! 🚀")
        break

    # ABSOLUTE PROTECTION LAYER: Completely strips anything not in training 'stoi'
    sanitized_prompt = "".join([c for c in raw_prompt if c in stoi])

    if len(sanitized_prompt) == 0:
        print("⚠️ Warning: Empty input after filtering. Defaulting to newline.")
        sanitized_prompt = "\n"
    elif len(sanitized_prompt) < len(raw_prompt):
        dropped = set(raw_prompt) - set(sanitized_prompt)
        print(f"⚠️ Safely filtered away unseen characters: {dropped}")

    user_temp = input("🔥 Enter Temperature (0.1 to 1.2) [Default 0.8]: ")
    try:
        temp_val = float(user_temp) if user_temp.strip() else 0.8
        temp_val = max(0.1, min(temp_val, 1.2)) # Hard capped at 1.2 for math stability
    except ValueError:
        temp_val = 0.8

    token_count = input("🔢 Enter Tokens to Generate [Default 250]: ")
    try:
        tokens_val = int(token_count) if token_count.strip() else 250
        tokens_val = max(10, min(tokens_val, 1000))
    except ValueError:
        tokens_val = 250

    print(f"\n⚙️ Synthesizing {tokens_val} tokens via KV-Cache arrays...\n")
    print("." * 40)

    context_tensor = torch.tensor([encode(sanitized_prompt)], dtype=torch.long, device=device)

    try:
        generated_indices = m.generate_cached(context_tensor, max_new_tokens=tokens_val, temperature=temp_val)
        print(decode(generated_indices[0].tolist()))
    except RuntimeError as e:
        print(f"\n❌ Hardware tracking error: {e}")
        print("Please reset your kernel session.")
        break

    print("." * 40)


🚀 SYSTEM INITIALIZATION: Running execution node on [CUDA]
📥 Fetching Tiny Shakespeare dataset from source...

🏋️ STARTING ACCELERATED TRAINING LOOP...
   Step    0: Train Loss 4.2846 | Val Loss 4.2820
   Step  500: Train Loss 1.8879 | Val Loss 2.0031
   Step 1000: Train Loss 1.5370 | Val Loss 1.7278
   Step 1500: Train Loss 1.3935 | Val Loss 1.6118
   Step 2000: Train Loss 1.3078 | Val Loss 1.5483
   Step 2500: Train Loss 1.2496 | Val Loss 1.5182
   Step 2999: Train Loss 1.2029 | Val Loss 1.4993
✓ Training complete in 15.94 minutes.
💾 Weights saved successfully to local directory as: ./gpt_shakespeare_10.7M.pth

🧮 RUNNING STANDALONE PERFORMANCE SWEEP...

             PRODUCTION ARCHIVE METRICS REPORT
Mean Validation Loss:       1.4988
Model Perplexity (PPL):     4.48

             🎯 CRASH-PROOF CHAT INTERFACE ACTIVATED

Instructions: Type your prompt block below. Out-of-vocabulary text
will be automatically sanitized. Type 'exit' to terminate.

-----------------------------------------